This notebook:
- Builds input.json files (data points only) from dataset labels.
- Builds ground-truth error bar JSON from label distances.
- Runs CV-based detection to generate predictions.
- Compares predictions vs ground truth and reports accuracy/metrics.


In [14]:
# Install dependencies
%pip install numpy Pillow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\sulta\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [15]:
from pathlib import Path
import json
import math
import numpy as np
from PIL import Image

In [ ]:
# Config
from pathlib import Path

ROOT_DIR = Path.cwd().parent   # BarSight/
DATASET_DIR = ROOT_DIR / "dataset"
IMAGES_DIR = DATASET_DIR / "images"
LABELS_DIR = DATASET_DIR / "labels"

INPUT_DIR = Path("../Input_jsons")
GT_DIR = Path("../Error_bar_groundTruth")
PRED_DIR = Path("../Error_bar_prediction")

INPUT_DIR.mkdir(parents=True, exist_ok=True)
GT_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)

INPUT_JSON = LABELS_DIR / "0b9bbea3-b480-4237-92ca-a63b9107f9e2.json"  # label format
IMAGE_PATH = IMAGES_DIR / "0b9bbea3-b480-4237-92ca-a63b9107f9e2.png"
OUTPUT_JSON = Path("detected_error_bars.json")

# Detection parameters
COLOR_TOL = 45.0        # color distance tolerance
GAP_LIMIT = 2           # allowed gaps in scan (pixels)
WINDOW = 2              # horizontal scan window (pixels)


IMAGE_DIR = IMAGES_DIR


In [40]:
# Debug paths
print('CWD:', Path.cwd())
print('LABELS_DIR:', LABELS_DIR)
print('Labels count:', len(list(LABELS_DIR.glob('*.json'))))


CWD: c:\Users\sulta\Documents\BarSight\notebooks
LABELS_DIR: c:\Users\sulta\Documents\BarSight\dataset\labels
Labels count: 150


In [41]:
def load_label_file(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))

def build_input_json(label_data, image_file):
    data_points = []
    for entry in label_data:
        line_name = entry.get("label", {}).get("lineName", "")
        points = []
        for point in entry.get("points", []):
            if point.get("label"):
                continue
            points.append({"x": point["x"], "y": point["y"]})
        if points:
            data_points.append({"lineName": line_name, "points": points})

    return {"image_file": image_file, "data_points": data_points}

def build_ground_truth(label_data, image_file):
    error_bars = []
    for entry in label_data:
        line_name = entry.get("label", {}).get("lineName", "")
        points = []
        for point in entry.get("points", []):
            if point.get("label"):
                continue
            x, y = point["x"], point["y"]
            top = point.get("topBarPixelDistance", 0)
            bottom = point.get("bottomBarPixelDistance", 0)
            upper_y = y - top
            lower_y = y + bottom
            points.append({
                "data_point": {"x": float(x), "y": float(y)},
                "upper_error_bar": {"x": float(x), "y": float(upper_y)},
                "lower_error_bar": {"x": float(x), "y": float(lower_y)},
            })
        if points:
            error_bars.append({"lineName": line_name, "points": points})

    return {"image_file": image_file, "error_bars": error_bars}


In [42]:
def _dominant_color(patch):
    flat = patch.reshape(-1, 3)
    non_white = flat[(flat < 245).any(axis=1)]
    if len(non_white) == 0:
        return np.median(flat, axis=0)
    return np.median(non_white, axis=0)

def _find_endpoint(image, x, y_start, direction, target_color, tol, gap_limit, window):
    height, width, _ = image.shape
    x = int(round(x))
    x_min = max(0, x - window)
    x_max = min(width - 1, x + window)

    last_match = None
    gap = 0
    y = int(round(y_start))
    while 0 <= y < height:
        strip = image[y, x_min : x_max + 1]
        dist = np.linalg.norm(strip.astype(np.float32) - target_color, axis=1)
        if np.any(dist <= tol):
            last_match = y
            gap = 0
        else:
            if last_match is not None:
                gap += 1
                if gap >= gap_limit:
                    break
        y += direction
    return last_match

def detect_error_bars(image, line_groups, tol=45.0, gap_limit=2, window=2):
    output = []
    for line in line_groups:
        line_name = line.get("lineName", "")
        points_output = []
        for point in line.get("points", []):
            x, y = point["x"], point["y"]
            y_int = int(round(y))
            x_int = int(round(x))

            y0 = max(0, y_int - 2)
            y1 = min(image.shape[0], y_int + 3)
            x0 = max(0, x_int - 2)
            x1 = min(image.shape[1], x_int + 3)
            patch = image[y0:y1, x0:x1]
            if patch.size == 0:
                target = np.array([0, 0, 0], dtype=np.float32)
            else:
                target = _dominant_color(patch)

            up = _find_endpoint(image, x, y, -1, target, tol, gap_limit, window)
            down = _find_endpoint(image, x, y, 1, target, tol, gap_limit, window)

            if up is None:
                up = y
            if down is None:
                down = y

            points_output.append({
                "data_point": {"x": float(x), "y": float(y)},
                "upper_error_bar": {"x": float(x), "y": float(up)},
                "lower_error_bar": {"x": float(x), "y": float(down)},
            })
        output.append({"lineName": line_name, "points": points_output})
    return output


In [43]:
# Build input.json and ground-truth for all labels
label_files = sorted(LABELS_DIR.glob("*.json"))
print(f"Found {len(label_files)} label files")

for idx, label_path in enumerate(label_files, start=1):
    label_data = load_label_file(label_path)
    image_file = label_path.stem + ".png"

    input_json = build_input_json(label_data, image_file)
    gt_json = build_ground_truth(label_data, image_file)

    (INPUT_DIR / f"{label_path.stem}.json").write_text(json.dumps(input_json, indent=2), encoding="utf-8")
    (GT_DIR / f"{label_path.stem}.json").write_text(json.dumps(gt_json, indent=2), encoding="utf-8")

    if idx % 50 == 0:
        print(f"Prepared {idx}/{len(label_files)}")

print("Input + GroundTruth generation done.")


Found 150 label files
Prepared 50/150
Prepared 100/150
Prepared 150/150
Input + GroundTruth generation done.


In [44]:
# Predict error bars for all inputs
input_files = sorted(INPUT_DIR.glob("*.json"))
print(f"Found {len(input_files)} input files")

for idx, input_path in enumerate(input_files, start=1):
    data = json.loads(input_path.read_text(encoding="utf-8"))
    image_file = data.get("image_file")
    line_groups = data.get("data_points", [])

    img_path = IMAGES_DIR / image_file
    if not img_path.exists():
        print(f"[skip] image not found for {input_path.name}")
        continue

    image = np.array(Image.open(img_path).convert("RGB"))
    output = {
        "image_file": image_file,
        "error_bars": detect_error_bars(image, line_groups, tol=COLOR_TOL, gap_limit=GAP_LIMIT, window=WINDOW),
    }

    (PRED_DIR / f"{input_path.stem}.json").write_text(json.dumps(output, indent=2), encoding="utf-8")

    if idx % 50 == 0:
        print(f"Predicted {idx}/{len(input_files)}")

print("Prediction done.")


Found 150 input files
Predicted 50/150
Predicted 100/150
Predicted 150/150
Prediction done.


In [46]:
PIX_TOL = 3.0  # pixel tolerance for matching

In [47]:
# Evaluate predictions vs ground truth
def flatten_points(error_bars):
    flat = []
    for line in error_bars:
        for pt in line.get("points", []):
            flat.append(pt)
    return flat

tp = 0
fp = 0
fn = 0
abs_err_up = []
abs_err_down = []

gt_files = sorted(GT_DIR.glob("*.json"))
for gt_path in gt_files:
    pred_path = PRED_DIR / gt_path.name
    if not pred_path.exists():
        continue

    gt = json.loads(gt_path.read_text(encoding="utf-8"))
    pred = json.loads(pred_path.read_text(encoding="utf-8"))

    gt_points = flatten_points(gt.get("error_bars", []))
    pred_points = flatten_points(pred.get("error_bars", []))

    # assume same ordering per line/point
    for g, p in zip(gt_points, pred_points):
        gu = g["upper_error_bar"]["y"]
        gl = g["lower_error_bar"]["y"]
        pu = p["upper_error_bar"]["y"]
        pl = p["lower_error_bar"]["y"]

        abs_err_up.append(abs(gu - pu))
        abs_err_down.append(abs(gl - pl))

        ok = (abs(gu - pu) <= PIX_TOL) and (abs(gl - pl) <= PIX_TOL)
        if ok:
            tp += 1
        else:
            fp += 1

# Confusion-style summary (per point)
total = tp + fp
accuracy = tp / total if total else 0
mae_up = float(np.mean(abs_err_up)) if abs_err_up else 0.0
mae_down = float(np.mean(abs_err_down)) if abs_err_down else 0.0

results = {
    "points_total": total,
    "points_correct_within_tol": tp,
    "points_incorrect": fp,
    "accuracy_within_tol": accuracy,
    "mae_upper": mae_up,
    "mae_lower": mae_down,
    "pixel_tolerance": PIX_TOL,
}
results


{'points_total': 4839,
 'points_correct_within_tol': 1147,
 'points_incorrect': 3692,
 'accuracy_within_tol': 0.23703244471998347,
 'mae_upper': 25.123892197113555,
 'mae_lower': 24.95102585928372,
 'pixel_tolerance': 3.0}